# Reproduce the corrected small-data model-selection benchmark
**RSI2026510 | AI Tools for Research, Implementation Paper**

This notebook executes the real project benchmark rather than displaying sample or expected scores. It requires the ZIP project archive supplied with the paper. The supported experiment uses three primary tabular datasets and one image-derived supplementary task, two seeds, nested 3×3 stratified validation, and fixed/search baselines. It does **not** implement FLAML or an LLM agent.

**Important:** The saved CSVs in the ZIP are output from a prior real run; running the source again overwrites them. Review the student-owned execution output and retain your own logs as new evidence.

In [ ]:
# Upload the accompanying ZIP in Google Colab, or locate it locally.
import os, sys, zipfile, subprocess
from pathlib import Path
archive_name = 'AITR_Lab02_RSI2026510_Corrected.zip'
try:
    from google.colab import files
    print('Please upload the accompanying project ZIP: ', archive_name)
    uploaded = files.upload()
    candidate = [Path(k) for k in uploaded if k.lower().endswith('.zip')]
    if not candidate: raise FileNotFoundError('No ZIP archive uploaded')
    archive_path=candidate[0]
except ImportError:
    archive_path=Path.cwd()/archive_name
    if not archive_path.exists():
        archive_path=Path('/mnt/data')/archive_name
assert archive_path.exists(), f'Missing project archive: {archive_path}'
working_dir=Path('/content') if Path('/content').exists() else Path.cwd()
with zipfile.ZipFile(archive_path) as z:
    z.extractall(working_dir)
project=working_dir/'AITR_Lab02_RSI2026510_Corrected'
assert (project/'src'/'benchmark.py').exists(), project
print('Extracted reproducible project to:',project)

In [ ]:
# Install dependencies using the included requirements.
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(project/'requirements.txt')],check=True)
import sklearn,optuna,pandas,numpy,matplotlib
print('Python',sys.version.split()[0], '| sklearn',sklearn.__version__,'| optuna',optuna.__version__)

In [ ]:
# Inspect the actual bundled dataset profile and original run records.
from IPython.display import display
import pandas as pd
display(pd.read_csv(project/'results'/'dataset_profiles.csv'))
print('Saved original run summary (not a substitute for a fresh run):')
display(pd.read_csv(project/'results'/'tabular_only_summary.csv'))

In [ ]:
# RUN THE ACTUAL EXPERIMENT. This recreates the result CSVs from scratch.
# Depending on hardware and package versions, measured runtime and marginal scores can differ.
env=os.environ.copy();env.update({'OPENBLAS_NUM_THREADS':'1','OMP_NUM_THREADS':'1','MKL_NUM_THREADS':'1'})
cmd=[sys.executable,str(project/'src'/'benchmark.py'),'--seeds','42','123','--trials','12']
print('Executing:', ' '.join(cmd),flush=True)
subprocess.run(cmd,env=env,cwd=project,check=True)

In [ ]:
# Verify recorded predictions against every reported metric and test-fold alignment.
subprocess.run([sys.executable,str(project/'src'/'verify_outputs.py')],cwd=project,check=True)
subprocess.run([sys.executable,str(project/'src'/'plots.py')],cwd=project,check=True)

In [ ]:
# Display results calculated by the ACTUAL new run.
print('PRIMARY: three genuine tabular tasks')
display(pd.read_csv(project/'results'/'tabular_only_summary.csv'))
print('FOUR TASKS: includes supplementary flattened-image digit vectors')
display(pd.read_csv(project/'results'/'overall_summary.csv'))
print('INDEPENDENT METRIC VERIFICATION:')
print((project/'results'/'verification_report.json').read_text())

In [ ]:
# Display figures rebuilt from results CSVs.
from IPython.display import display,Image
for figure in ['per_dataset_accuracy.png','runtime_tradeoff.png','budget_effect.png']:
    print(figure)
    display(Image(filename=str(project/'results'/'figures'/figure),width=760))

## Research-integrity note
An executed cell is evidence of **this** Colab run, not proof of global near-optimal performance. Record your own package versions, warnings, and measured scores. Keep test folds out of model selection, and do not alter search settings after looking at outer-test results. Update the first-person AI assistance log after personally inspecting and explaining the code.

The instructor also requires four external institution-accepted reports: similarity/plagiarism and AI-writing reports for **both** the review and implementation papers. Those cannot be generated from this notebook.